# Native CLM v0 M2-R0b — Numerical Reference Audit

Numerical/optimizer-mechanics diagnostic only. It consumes no new continual-language formal seeds and does not change the historical M2 or M2-R0 decisions. R0b separates the projected-gradient analytic update, parameter-dtype `fl(W + Delta) - W` floor, optimizer-realized update, and actually committed update.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m2r0b-numerical-reference-audit'
REPO = Path('/kaggle/working/mini-cells')
M1_DIR = Path('/kaggle/working/native-clm-v0-m1')
M1 = M1_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/native-clm-m2r0-data')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m2r0b-numerical-reference-audit'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for canonical M2-R0b'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Exact canonical M1 checkpoint; SHA verification is authoritative.
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--revision', '2b6ac153e926f899f038ff02c8c10041baaacb4a',
    '--output', M1,
])
print((M1_DIR / 'provenance.json').read_text())

In [ ]:
# Reconstruct the same exact pinned WikiText B gradient source used by M2-R0.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m2r0_data.py', '--output-dir', DATA])
manifest = json.loads((DATA / 'manifest.json').read_text())
b = manifest['files']['B_train']
print(json.dumps({'format': manifest['format'], 'B_revision': manifest['dataset_revisions']['B'], 'B_train': b}, indent=2))
assert manifest['format'] == 'minicells.native-clm-v0.m2r0-data-manifest.v1'
assert b['sha256'] == 'c7029622b9a1c4b4f249d927b3ab30d4c09ddffe845e031aee3412b4735ca440'

In [ ]:
# Five frozen optimizer arms, identical data-order seed, no formal continual-language run.
run([
    sys.executable, 'scripts/research/run_native_clm_v0_m2r0b.py',
    '--checkpoint', M1,
    '--data-dir', DATA,
    '--output-dir', OUT,
    '--device', 'cuda',
])
result = json.loads((OUT / 'diagnostic-result.json').read_text())
print(json.dumps({
    'classification': result['classification'],
    'mechanics_diagnosis': result['mechanics_diagnosis'],
    'm2_r1_unblocked': result['m2_r1_unblocked'],
    'scientific_decision': result['scientific_decision'],
    'new_formal_seeds_consumed': result['new_formal_seeds_consumed'],
}, indent=2))
for arm, summary in result['arms'].items():
    print(arm, 'n=', summary['audited_cell_updates'], 'rho_p95=', summary['committed_rho_p95'], 'excess_p95=', summary['committed_excess_factor_p95'], 'excess_max=', summary['committed_excess_factor_max'])

In [ ]:
# Publish lightweight JSON/CSV/MD evidence only. No checkpoint is generated.
run([
    sys.executable, 'scripts/research/publish_native_clm_v0_m2r0b.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
])
print('Published M2-R0b classification:', result['classification'])